In [ ]:
import numpy as np
import pandas as pd

Un système de recommandation est un système intelligent qui essaye deprédire les préférences des utilisateurs sur certains produits. L' objectif est de trouver une relation entre l'utilisateur et les produits afin de maximiser l'engagement utilisateur-produit.

L'application principale des systèmes de recommandation consiste à suggérer une vidéo (une chanson, un film, ...) à l'utilisateur associée aux éléments aimés dans le passé et qui répond à ses gouts.
Ils sont également utilisé pour recommander des contenus en fonction des comportements des utilisateurs sur les réseaux sociaux et les sites d'actualités.

Les deux approches basiques pour créer des systèmes de recommandation à l'aide du machine learning sont les suivants:
- Filtrage Collaboratif : L'hypothèse de cette approche est que les personnes qui ont aimé un article dans le passé aimeront également le même à l'avenir. Cette approche construit un modèle basé sur le comportement passé des utilisateurs. Le comportement de l'utilisateur peut inclure des vidéos visionnées précédemment, des articles achetés, des évaluations données sur des articles. De cette manière, le modèle trouve une association entre les utilisateurs et les éléments. Le modèle est ensuite utilisé pour prédire l'élément susceptible d'intéresser l'utilisateur. La décomposition en valeurs singulières (SVD) est une des techniques utilisée comme approche de filtrage collaboratif dans les systèmes de recommandation.

- Filtrage basé sur le contenu : cette démarche s’appuie sur une description de l’article et un enregistrement des préférences des utilisateurs. Il utilise une séquence de caractéristiques discrètes et pré-étiquetées d'un article afin de recommander des articles supplémentaires avec des propriétés similaires. Cette approche est la mieux adaptée lorsqu'il existe suffisamment d'informations disponibles sur les éléments, mais pas sur les préférences des utilisateurs.

Dans ce TP, nous allons nous concentrer sur la construction d'un système de recommendation de films de type filtrage collaboratif à l'aide de la technique d'algèbre linéaire SVD.

# Lecture données

Téléchargez le dataset "MovieLens 1M movie ratings": https://grouplens.org/datasets/movielens/1m/ . Il s'agit d'un dataset contenant les notes de 1 à 5 données par différents utilisateurs à des films de type différent.

Ensuite importez le ici à l'aide de Pandas. Ce dataset ne demande pas de prétraitement ni de préparation des données.

In [17]:
# TODO inserez le path vers les données
rating_data_path = r"C:\Users\benoi\Desktop\YNOV\semaine_15_06_2026\archive\rating.csv"
movie_data_path = r"C:\Users\benoi\Desktop\YNOV\semaine_15_06_2026\archive\anime.csv"

In [18]:
rating_data = pd.read_csv(rating_data_path, 
    engine='python')

# Filtrer les ratings valides (ignorer les -1 qui signifient "pas noté")
rating_data = rating_data[rating_data['rating'] > 0]

In [19]:
movie_data = pd.read_csv(movie_data_path,
    engine='python', encoding='latin-1')

# Garder seulement les colonnes pertinentes
movie_data = movie_data[['anime_id', 'name', 'genre']]
movie_data = movie_data.rename(columns={'name': 'title'})

In [20]:
# Visualisation d'exemples du dataset rating_data
print("Premières lignes du dataset rating_data:")
print(rating_data.head(10))
print("\nInformations sur le dataset:")
print(rating_data.info())
print("\nStatistiques du dataset:")
print(rating_data.describe())

Premières lignes du dataset rating_data:
     user_id  anime_id  rating
47         1      8074      10
81         1     11617      10
83         1     11757      10
101        1     15451      10
153        2     11771      10
156        3        20       8
157        3       154       6
158        3       170       9
159        3       199      10
160        3       225       9

Informations sur le dataset:
<class 'pandas.core.frame.DataFrame'>
Index: 6337241 entries, 47 to 7813736
Data columns (total 3 columns):
 #   Column    Dtype
---  ------    -----
 0   user_id   int64
 1   anime_id  int64
 2   rating    int64
dtypes: int64(3)
memory usage: 193.4 MB
None

Statistiques du dataset:
            user_id      anime_id        rating
count  6.337241e+06  6.337241e+06  6.337241e+06
mean   3.674791e+04  8.902866e+03  7.808497e+00
std    2.101340e+04  8.882000e+03  1.572496e+00
min    1.000000e+00  1.000000e+00  1.000000e+00
25%    1.898400e+04  1.239000e+03  7.000000e+00
50%    3.681500e

In [21]:
# Visualisation d'exemples du dataset movie_data
print("Premières lignes du dataset movie_data:")
print(movie_data.head(10))
print("\nInformations sur le dataset:")
print(movie_data.info())
print("\nNombre total de films:")
print(len(movie_data))

Premières lignes du dataset movie_data:
   anime_id                                              title  \
0     32281                                     Kimi no Na wa.   
1      5114                   Fullmetal Alchemist: Brotherhood   
2     28977                                          GintamaÂ°   
3      9253                                        Steins;Gate   
4      9969                                      Gintama&#039;   
5     32935  Haikyuu!!: Karasuno Koukou VS Shiratorizawa Ga...   
6     11061                             Hunter x Hunter (2011)   
7       820                               Ginga Eiyuu Densetsu   
8     15335  Gintama Movie: Kanketsu-hen - Yorozuya yo Eien...   
9     15417                           Gintama&#039;: Enchousen   

                                               genre  
0               Drama, Romance, School, Supernatural  
1  Action, Adventure, Drama, Fantasy, Magic, Mili...  
2  Action, Comedy, Historical, Parody, Samurai, S...  
3            

TODO Optionnel : analysez le dataset et visualisez ses caractéristiques et des stats simples liées à son contenu:
- quel est le film le plus populaire en moyenne?
- quels sont les films les plus notés?
- quelle est la note moyenne des films?
- quel est l'utilisateur qui a noté le plus de films? combien?
- ...

In [22]:
# Vérification des données
print("Vérification du chargement des données:")
print(f"Nombre de lignes dans rating_data: {len(rating_data)}")
print(f"Colonnes de rating_data: {rating_data.columns.tolist()}")
print(f"Premières valeurs de rating_data:\n{rating_data.head()}")

print(f"\n--- INFO MOVIE DATA ---")
print(f"Nombre de films: {len(movie_data)}")
print(f"Colonnes de movie_data: {movie_data.columns.tolist()}")
print(f"Premières valeurs de movie_data:\n{movie_data.head()}")

# Analyse optionnelle du dataset
if len(rating_data) > 0 and len(movie_data) > 0:
    # 1. Quel est le film le plus populaire en moyenne?
    avg_ratings = rating_data.groupby('anime_id')['rating'].mean()
    if len(avg_ratings) > 0:
        most_popular_anime_id = avg_ratings.idxmax()
        most_popular_avg = avg_ratings.max()
        most_popular_title = movie_data[movie_data['anime_id'] == most_popular_anime_id]['title'].values[0]
        print(f"\nFilm le plus populaire en moyenne: {most_popular_title} (note moyenne: {most_popular_avg:.2f})")
    
    # 2. Quels sont les films les plus notés?
    rating_counts = rating_data.groupby('anime_id').size().sort_values(ascending=False)
    print(f"\nTop 10 films les plus notés:")
    for i, (anime_id, count) in enumerate(rating_counts.head(10).items(), 1):
        title = movie_data[movie_data['anime_id'] == anime_id]['title'].values[0] if len(movie_data[movie_data['anime_id'] == anime_id]) > 0 else f"ID {anime_id}"
        print(f"  {i}. {title}: {count} notes")
    
    # 3. Quelle est la note moyenne des films?
    overall_avg = rating_data['rating'].mean()
    print(f"\nNote moyenne globale de tous les films: {overall_avg:.2f}")
    
    # 4. Quel est l'utilisateur qui a noté le plus de films?
    user_rating_counts = rating_data.groupby('user_id').size().sort_values(ascending=False)
    most_active_user = user_rating_counts.index[0]
    most_active_count = user_rating_counts.iloc[0]
    print(f"Utilisateur le plus actif: ID {most_active_user} (a noté {most_active_count} films)")
    
    # Stats supplémentaires
    print(f"\nStats supplémentaires:")
    print(f"- Nombre total d'utilisateurs: {rating_data['user_id'].nunique()}")
    print(f"- Nombre total de films: {rating_data['anime_id'].nunique()}")
    print(f"- Nombre total de notes: {len(rating_data)}")
    print(f"- Distribution des notes: \n{rating_data['rating'].value_counts().sort_index()}")
else:
    print("\nErreur: Les données n'ont pas été correctement chargées!")

Vérification du chargement des données:
Nombre de lignes dans rating_data: 6337241
Colonnes de rating_data: ['user_id', 'anime_id', 'rating']
Premières valeurs de rating_data:
     user_id  anime_id  rating
47         1      8074      10
81         1     11617      10
83         1     11757      10
101        1     15451      10
153        2     11771      10

--- INFO MOVIE DATA ---
Nombre de films: 12294
Colonnes de movie_data: ['anime_id', 'title', 'genre']
Premières valeurs de movie_data:
   anime_id                             title  \
0     32281                    Kimi no Na wa.   
1      5114  Fullmetal Alchemist: Brotherhood   
2     28977                         GintamaÂ°   
3      9253                       Steins;Gate   
4      9969                     Gintama&#039;   

                                               genre  
0               Drama, Romance, School, Supernatural  
1  Action, Adventure, Drama, Fantasy, Magic, Mili...  
2  Action, Comedy, Historical, Parody, Sam

# Modèle

La décomposition en valeurs singulières (SVD) est une méthode d'algèbre linéaire qui est utilisée comme technique de réduction de dimensionnalité dans l'apprentissage automatique (et dans d'autres domaines).
La SVD est une technique de factorisation matricielle, qui décompose une matrice en d'autres matrices plus simples, et elle est utilisée pour réduire le nombre de features d'un ensemble de données en réduisant la dimension spatiale de la dimension $N$ à la dimension $K$.

Dans le cadre des système de recommandation, la SVD est utilisé comme technique de filtrage collaboratif.
Elle se base sur l'utilisation d'une matrice $A$ où chaque ligne représente un utilisateur et chaque colonne représente un élément (film, livre, produit, ...). Les valeurs de cette matrice sont les notes attribuées aux éléments par les utilisateurs.


La matrice $A$ est factorisée à l'aide de la SVD:
$A = U S V^T$

- $U$ de taille représente la relation entre les utilisateurs et les facteurs latents du nouvel espace vectoriel
- $V$ représente la relation entre les éléments et les facteurs latents du nouvel espace vectoriel
- $S$ décrit l'intensité de chaque facteur latent

Dans la SVD, les colonnes de la matrice $U$ sont des vecteurs propres de $A A^T$ et les lignes de la matrice $V$ sont des vecteurs propres de $A^T A$. Ce qui est intéressant, c'est que $A^T A$ et $A A^T$ sont potentiellement de taille différente (car la matrice $A$ peut être de forme non carrée), mais ils ont le même ensemble de valeurs propres, qui sont le carré des valeurs sur la diagonale de $S$.
C'est pourquoi le résultat de la SVD peut révéler beaucoup de choses sur la matrice $A$.

La SVD réduit la dimension de la matrice $A$ en extrayant ses facteurs latents et en gardans seulement ceux avec importantece éléveé. Les facteurs latents ici représentent des caractéristiques des éléments, par exemple, le genre de musique ou de film, mais parfois ils sont un peu complèxes à interpréter par un humain. 
Cette opération cartographie chaque utilisateur et chaque élément dans un espace latent à $K$ dimensions. Ce mappage facilite une représentation claire des relations entre les utilisateurs et les éléments.

Imaginez que nous ayons collecté dans $A$ des notes de films de manière que les films sont des colonnes et les utilisateurs sont des lignes, et les éléments de la matrice sont les notes qu'une personne a données à un film.
Dans ce cas, $A A^T$ est un tableau personne-personne, où l'on peut retrouver les liens de similarité entre les notes de différentes personnes. 
De même, $A^T A$ est un tableau film-film dont les éléments contiennet la similarité des notes entre films différents.

In [23]:
# fonction pour trouver l'id d'un anime sachant son titre
# fonctionne seulement si le titre est dans la base de données

def find_id_given_title(given_title):
    # match title with the one in the database
    matching_title = np.unique([title for title in movie_data.title if given_title in title])[0]
    # find id number
    return movie_data[movie_data.title == matching_title].anime_id.values[0]

In [24]:
# fonction qui calcule la cosine similarity entre un anime donné et les autres du dataset
# et qui donne en sortie les id des 10 animes les plus similaires sur la base des notes des utilisateurs.
# cette similarité peut être calculée sur les features extraites par la SVD (les vecteurs du nouvel espace V)

def top_cosine_similarity(data, movie_idx, top_n=10):
    index = movie_idx  # position dans la matrice (pas l'anime_id)
    movie_row = data[index, :]
    magnitude = np.sqrt(np.einsum('ij, ij -> i', data, data))
    similarity = np.dot(movie_row, data.T) / (magnitude[index] * magnitude)
    sort_indexes = np.argsort(-similarity)
    return sort_indexes[1:top_n+1]


In [25]:
# fonction pour imprimer les titres des top 10 animes les plus similaires à un anime donné

def print_similar_movies(movie_data, anime_id, V, top_n=10, k=50):
    
    sliced = V.T[:, :k]  # utilisation seulement des K features latentes les plus importantes
    movie_idx = anime_ids_index.index(anime_id)  # convertir anime_id → position dans la matrice
    top_indexes = top_cosine_similarity(sliced, movie_idx, top_n)
    
    print('Recommendations for {0}: \n'.format(
        movie_data[movie_data.anime_id == anime_id].title.values[0]))
    for pos in top_indexes:
        aid = anime_ids_index[pos]  # convertir position → anime_id réel
        print(movie_data[movie_data.anime_id == aid].title.values[0])


Création de la matrice $A$

In [26]:
# TODO créez la matrice ratings_mat contenant les notes des animes par utilisateur
# (animes sur les lignes, utilisateurs sur les colonnes)

# Créer un pivot table : animes en lignes, utilisateurs en colonnes
ratings_mat = rating_data.pivot_table(index='anime_id', columns='user_id', values='rating', fill_value=0)
print(f"Taille de la matrice ratings: {ratings_mat.shape}")
print(f"Nombre total d'animes: {ratings_mat.shape[0]}")
print(f"Nombre total d'utilisateurs: {ratings_mat.shape[1]}")

# Sauvegarder l'ordre des anime_ids avant conversion en numpy (les IDs ne sont pas contiguës)
anime_ids_index = ratings_mat.index.tolist()  # position i → anime_id réel
ratings_mat = ratings_mat.values  # Convertir en numpy array


Taille de la matrice ratings: (9927, 69600)
Nombre total d'animes: 9927
Nombre total d'utilisateurs: 69600


In [27]:
# normalisation de la matrice : on soustrait la moyenne
normalised_mat = ratings_mat - np.asarray([(np.mean(ratings_mat, 1))]).T
# ultérieure normalisation et transposition pour passer à la matrice A "classique"
A = normalised_mat.T / np.sqrt(ratings_mat.shape[0] - 1)

Calcul de la SVD avec numpy

In [28]:
# Calcul de la Singular Value Decomposition (SVD) à l'aide de numpy
U, S, Vh = np.linalg.svd(A, full_matrices=False)


In [29]:
print(f"Taille de U  : {U.shape}  → (n_utilisateurs × k facteurs latents)")
print(f"Taille de S  : {S.shape}  → (k valeurs singulières)")
print(f"Taille de Vh : {Vh.shape} → (k facteurs latents × n_animes)")
print()
print("Conclusion :")
print(f"  • U  : chaque utilisateur est représenté par {U.shape[1]} facteurs latents")
print(f"  • S  : {S.shape[0]} valeurs singulières classées par ordre décroissant d'importance")
print(f"  • Vh : chaque anime est représenté par {Vh.shape[0]} facteurs latents")
print(f"\nLes premières valeurs de S (les plus importantes) :")
print(S[:10])


Taille de U  : (69600, 9927)  → (n_utilisateurs × k facteurs latents)
Taille de S  : (9927,)  → (k valeurs singulières)
Taille de Vh : (9927, 9927) → (k facteurs latents × n_animes)

Conclusion :
  • U  : chaque utilisateur est représenté par 9927 facteurs latents
  • S  : 9927 valeurs singulières classées par ordre décroissant d'importance
  • Vh : chaque anime est représenté par 9927 facteurs latents

Les premières valeurs de S (les plus importantes) :
[6.86941190e-312 6.86940028e-312 1.13528736e+000 2.34770115e-001
 3.18534483e-002 2.38735632e-001 3.95531609e-001 6.10201149e-002
 1.80316092e-001 5.22844828e-001]


utilisation de la SVD pour la recommendation de films

In [33]:
# Anime de référence : choisissez un titre présent dans le dataset
anime_title = "Naruto"
anime_id = find_id_given_title(anime_title)

# Affichage des 10 animes les plus similaires (k=50 facteurs latents)
print_similar_movies(movie_data, anime_id, Vh, top_n=10, k=50)


Recommendations for Boruto: Naruto the Movie: 

FlashBack
One Off
Galaxy Angel Specials
Houkago no Tinker Bell
Usogui
Shounen Ninja Kaze no Fujimaru
Kyoto Animation: Hoshi-hen
Henshin Gattai! 5 tsu no Atsuki Tamashii
Crimson Girls: Chikan Shihai
Jormungand: Perfect Order


C:\Users\benoi\AppData\Local\Temp\ipykernel_36184\1027315862.py:9: RuntimeWarning: invalid value encountered in divide
  similarity = np.dot(movie_row, data.T) / (magnitude[index] * magnitude)


TODO optionnel :  Utilisez truncated SVD from Scikit. Pourquoi une approche tronquée ?

In [34]:
from sklearn.decomposition import TruncatedSVD

# La SVD tronquée ne calcule que les K premières valeurs singulières au lieu de toutes.
# Avantages :
#   - Beaucoup plus rapide sur des grandes matrices creuses (sparse)
#   - Consomme moins de mémoire
#   - Évite le bruit des petites valeurs singulières peu informatives
k = 50
svd = TruncatedSVD(n_components=k, random_state=42)

# fit_transform sur A.T (n_animes × n_utilisateurs) → représentation (n_animes × k)
Vh_truncated = svd.fit_transform(A.T)
print(f"Taille de la matrice transformée (SVD tronquée) : {Vh_truncated.shape}")
print(f"Variance expliquée cumulée par {k} composantes : {svd.explained_variance_ratio_.sum():.2%}")

# Recommandations avec la SVD tronquée
anime_id = find_id_given_title("Naruto")
movie_idx = anime_ids_index.index(anime_id)  # convertir anime_id → position dans la matrice
top_indexes = top_cosine_similarity(Vh_truncated, movie_idx, top_n=10)

print(f"\nRecommandations (SVD tronquée) pour {movie_data[movie_data.anime_id == anime_id].title.values[0]} :")
for pos in top_indexes:
    aid = anime_ids_index[pos]  # convertir position → anime_id réel
    title_row = movie_data[movie_data.anime_id == aid]
    if len(title_row) > 0:
        print(f"  • {title_row.title.values[0]}")


Taille de la matrice transformée (SVD tronquée) : (9927, 50)
Variance expliquée cumulée par 50 composantes : 36.11%

Recommandations (SVD tronquée) pour Boruto: Naruto the Movie :
  • The Last: Naruto the Movie
  • Boruto: Naruto the Movie - Naruto ga Hokage ni Natta Hi
  • Fairy Tail (2014)
  • Nanatsu no Taizai: Seisen no Shirushi
  • Naruto: Shippuuden Movie 6 - Road to Ninja
  • Magi: Sinbad no Bouken (TV)
  • Magi: Sinbad no Bouken
  • Tokyo Ghoul: &quot;Jack&quot;
  • Naruto Shippuuden: Sunny Side Battle
  • Nanatsu no Taizai OVA


# Références

https://machinelearningmastery.com/using-singular-value-decomposition-to-build-a-recommender-system/

http://snap.stanford.edu/class/cs246-2015/handouts.html